# MLflow, model registry & deployment

This demo walks through the solution of the lab of the same name: a small convolutional network (LeNet, in PyTorch) that classifies landscape pictures into 6 classes, whose training is tracked with MLflow, which is published to MLflow's model registry, then served by a FastAPI API packaged in a Docker image.

The code is that of the [shuuchuu/landscape-classifier](https://github.com/shuuchuu/landscape-classifier) repository, which the notebook clones; a cell starting with `%%writefile <file>` writes its content to that file. These files are those of the repository's [`solution-en` branch](https://github.com/shuuchuu/landscape-classifier/tree/solution-en). The MLflow server is that of the trainer's [m09/landscape-classifier](https://dagshub.com/m09/landscape-classifier) DagsHub repository; in the lab, each trainee has their own.

## Setting up the environment

In [1]:
# @title Setting up the environment
import pathlib
import site
import sysconfig
import warnings

# MLflow records the model's installed dependencies, not uv.lock's
%set_env MLFLOW_UV_AUTO_DETECT false
if pathlib.Path.cwd().name != "landscape-classifier":
  !git clone -b main-en https://github.com/shuuchuu/landscape-classifier.git
  %cd landscape-classifier
!pip install -q -U uv
# httpx2: the HTTP client Starlette's TestClient expects, httpx being deprecated there
!uv pip install --system -e ".[fastapi]" httpx2
# the project, importable without a restart
site.addsitedir(sysconfig.get_paths()["purelib"])
# PyTorch warns each time it loads an MLflow pt2 model, to no consequence
warnings.filterwarnings("ignore", message="The given buffer is not writable")

env: MLFLOW_UV_AUTO_DETECT=false
Cloning into 'landscape-classifier'...
done.
/content/landscape-classifier
Using Python 3.13.16 environment at: /usr
Resolved 201 packages in 243ms
Prepared 1 package in 3ms
Uninstalled 1 package in 3ms
Installed 73 packages in 466ms
 + agent-detector==2.0.0
 + aiohttp-retry==2.9.1
 + alembic==1.20.1
 + amqp==5.4.2
 + antlr4-python3-runtime==4.9.3
 + appdirs==1.4.4
 + asyncssh==2.24.1
 + billiard==4.3.2
 + celery==5.7.0
 + click-didyoumean==0.3.1
 + click-plugins==1.1.1.2
 + click-repl==0.4.1
 + colorama==0.4.6
 + configobj==5.0.9
 + databricks-sdk==0.162.0
 + detect-installer==0.2.1
 + dictdiffer==0.10.0
 + diskcache==5.6.3
 + dnspython==2.9.2
 + docker==7.2.0
 + dpath==2.2.0
 + dulwich==1.2.21
 + dvc==3.67.1
 + dvc-data==3.18.3
 + dvc-http==2.32.0
 + dvc-objects==5.3.1
 + dvc-render==1.0.2
 + dvc-studio-client==0.23.0
 + dvc-task==0.40.2
 + email-validator==2.3.0
 + fastapi-cli==0.0.32
 + fastapi-cloud-cli==0.26.0
 + fastar==0.12.0
 + flask-cors==6.0.

## Credentials for the MLflow server

The repository's DagsHub username. The token (DagsHub account settings, *Tokens*) comes from Colab's secrets (`DAGSHUB_TOKEN`) or from the environment variable of that name, otherwise it is typed in without being shown. Without a username, the demo runs on a local MLflow server (`sqlite:///mlflow.db`), without a web interface:

In [2]:
username = ""

In [3]:
# @title Run me
import os
from getpass import getpass

# Empty, except in the copy `deckz labs gpu` sends: get_secret reads it otherwise
token = ""


def get_secret(name):
  """The secret `name`: a Colab secret, an environment variable, or typed in."""
  try:
    from google.colab import userdata
  except ImportError:  # outside Colab: an environment variable
    value = os.environ.get(name)
  else:
    try:
      value = userdata.get(name)
    except (
      userdata.SecretNotFoundError,
      userdata.NotebookAccessError,
      userdata.TimeoutException,
    ):
      value = None
  return value or getpass(f"{name}: ")


if username:
  os.environ["MLFLOW_TRACKING_URI"] = (
    f"https://dagshub.com/{username}/landscape-classifier.mlflow"
  )
  os.environ["MLFLOW_TRACKING_USERNAME"] = username
  os.environ["MLFLOW_TRACKING_PASSWORD"] = token or get_secret("DAGSHUB_TOKEN")
else:  # without a DagsHub account: a local MLflow server, without a web interface
  os.environ["MLFLOW_TRACKING_URI"] = "sqlite:///mlflow.db"
print("MLflow server:", os.environ["MLFLOW_TRACKING_URI"])

MLflow server: https://dagshub.com/m09/landscape-classifier.mlflow


## Getting the data

The training images go to `train-data/`, the test images to `test-data/`, one subdirectory per class:

In [4]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh landscape
!mv landscape/seg_train train-data
!mv landscape/seg_test test-data

remote: Enumerating objects: 2, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 2 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (2/2), 2.96 KiB | 2.96 MiB/s, done.
remote: Enumerating objects: 24289, done.
remote: Counting objects: 100% (24289/24289), done.
remote: Compressing objects: 100% (24289/24289), done.
remote: Total 24289 (delta 0), reused 24289 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (24289/24289), 341.65 MiB | 18.52 MiB/s, done.
Updating files: 100% (24336/24336), done.
✓ landscape — 401M


## Tracking the training and publishing the model

The `train` function tracks each training as an MLflow run of the `landscape-classifier` experiment: hyperparameters, each epoch's loss, accuracy on the test images. At the end of the run, it publishes the model to the registry under the name `dev.ml.landscape-classifier`, with the class names in its metadata. MLflow exports the model's graph by running it on `input_example`, a batch of one image. It also declares the model's dependencies (`pip_requirements`): the installed PyTorch version, without its local label (`+cu128` on Colab), which pip wouldn't find on PyPI.

In [5]:
%%writefile src/landscape_classifier/train.py
"""Training and evaluation, tracked with MLflow."""

import mlflow
import torch
from torch import nn
from torch.utils.data import DataLoader

from landscape_classifier.data import IMAGE_SIZE, get_loader
from landscape_classifier.model import LeNet

MODEL_NAME = "dev.ml.landscape-classifier"


def _device(model: nn.Module) -> torch.device:
  return next(model.parameters()).device


def train_epoch(
  model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer
) -> float:
  """Train `model` for one epoch.

  Returns:
      The mean loss over the epoch.
  """
  device = _device(model)
  loss_fn = nn.CrossEntropyLoss()
  model.train()
  total, count = 0.0, 0
  for images, labels in loader:
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()
    loss = loss_fn(model(images), labels)
    loss.backward()
    optimizer.step()
    total += loss.item() * len(labels)
    count += len(labels)
  return total / count


def evaluate(model: nn.Module, loader: DataLoader) -> float:
  """Measure `model`'s accuracy on `loader`.

  Returns:
      The share of images whose class `model` predicts right.
  """
  device = _device(model)
  model.eval()
  correct, count = 0, 0
  with torch.inference_mode():
    for images, labels in loader:
      predicted = model(images.to(device)).argmax(dim=1).cpu()
      correct += int((predicted == labels).sum())
      count += len(labels)
  return correct / count


def predict(model: nn.Module, loader: DataLoader) -> torch.Tensor:
  """Predict the class of every image of `loader`.

  Returns:
      The indices of the predicted classes.
  """
  device = _device(model)
  model.eval()
  with torch.inference_mode():
    return torch.cat(
      [model(images.to(device)).argmax(dim=1).cpu() for images, _ in loader]
    )


def train(train_dir: str, test_dir: str, learning_rate: float, epochs: int) -> None:
  """Train a LeNet on `train_dir`'s images and evaluate it on `test_dir`'s.

  The run is tracked in MLflow's `landscape-classifier` experiment, and the
  model published to the registry as a new version of `MODEL_NAME`.
  """
  train_loader = get_loader(train_dir, IMAGE_SIZE, shuffle=True)
  test_loader = get_loader(test_dir, IMAGE_SIZE)
  model = LeNet()
  optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

  mlflow.set_experiment("landscape-classifier")
  with mlflow.start_run():
    mlflow.log_params({"learning_rate": learning_rate, "epochs": epochs})
    for epoch in range(epochs):
      loss = train_epoch(model, train_loader, optimizer)
      mlflow.log_metric("train_loss", loss, step=epoch)
      print(f"Epoch {epoch + 1}/{epochs}: train loss {loss:.3f}")
    accuracy = evaluate(model, test_loader)
    mlflow.log_metric("test_accuracy", accuracy)
    print(f"Test accuracy: {accuracy:.3f}")
    mlflow.pytorch.log_model(
      model,
      name="model",
      registered_model_name=MODEL_NAME,
      input_example=torch.zeros(1, 3, *IMAGE_SIZE).numpy(),
      metadata={"classes": train_loader.dataset.classes},
      # PyTorch's version without its local label (+cu128 on Colab), not on PyPI
      pip_requirements=["torch==" + torch.__version__.split("+")[0]],
    )

Overwriting src/landscape_classifier/train.py


## Training

The `landscape-classifier` command runs the training, 3 epochs over the 14,000 images (a few minutes: it runs on the CPU, even with a GPU). At the end of the run, MLflow checks that the published model accepts `input_example` by loading it again: `MLFLOW_DEFAULT_PREDICTION_DEVICE=cpu` loads it on the CPU, where it was trained, even on a machine with a GPU:

In [6]:
# PYTHONWARNINGS and TORCH_LOGS: without torch.export's warnings, harmless here
!PYTHONWARNINGS=ignore TORCH_LOGS=-export MLFLOW_DEFAULT_PREDICTION_DEVICE=cpu landscape-classifier

2026/11/06 22:00:49 INFO mlflow.tracking.fluent: Experiment with name 'landscape-classifier' does not exist. Creating a new experiment.
Epoch 1/3: train loss 1.272
Epoch 2/3: train loss 0.991
Epoch 3/3: train loss 0.902
Test accuracy: 0.633
Successfully registered model 'dev.ml.landscape-classifier'.
2026/11/06 22:02:22 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: dev.ml.landscape-classifier, version 1
Created version '1' of model 'dev.ml.landscape-classifier'.
🏃 View run fun-calf-0 at: https://dagshub.com/m09/landscape-classifier.mlflow/#/experiments/0/runs/d635b6d9ab2f4218ae17087531e884d6
🧪 View experiment at: https://dagshub.com/m09/landscape-classifier.mlflow/#/experiments/0


Then open MLflow's interface, <https://dagshub.com/m09/landscape-classifier.mlflow>: the `landscape-classifier` experiment shows your run, its parameters and the `train_loss` curve, and the *Models* page the new version of `dev.ml.landscape-classifier`.

## Using the MLflow model registry

### Creating an MLflow client

The tracking URI does come from the `MLFLOW_TRACKING_URI` environment variable set at the top of the notebook:

In [7]:
import mlflow

MODEL_NAME = "dev.ml.landscape-classifier"

client = mlflow.MlflowClient()
client.tracking_uri

'https://dagshub.com/m09/landscape-classifier.mlflow'

### Searching models

Every model of the registry, each with the number of its most recent version:

In [8]:
def latest_version(name: str) -> str:
  versions = client.search_model_versions(f"name = '{name}'")
  return str(max(int(v.version) for v in versions))


for registered_model in client.search_registered_models():
  print(registered_model.name, latest_version(registered_model.name))

dev.ml.landscape-classifier 1


### Loading a model from the registry

We load the latest version with `mlflow.pytorch.load_model`, then predict the class of `forest1.jpg`, transformed by `get_transform`, the training's own function; the class names come from the model's metadata. The same model also loads as a `pyfunc` model, whose `predict` takes a NumPy array:

In [9]:
import torch
from PIL import Image

from landscape_classifier.data import IMAGE_SIZE, get_transform

version = latest_version(MODEL_NAME)
uri = f"models:/{MODEL_NAME}/{version}"
model = mlflow.pytorch.load_model(uri)
info = mlflow.models.get_model_info(uri)
print(list(info.flavors))

image = Image.open("forest1.jpg").convert("RGB")
batch = get_transform(IMAGE_SIZE)(image).unsqueeze(0)
with torch.inference_mode():
  probabilities = model(batch).softmax(dim=1)[0]
print(dict(zip(info.metadata["classes"], probabilities.tolist(), strict=True)))

pyfunc_model = mlflow.pyfunc.load_model(uri, model_config={"device": "cpu"})
pyfunc_model.predict(batch.numpy())

['python_function', 'pytorch']
{'buildings': 0.004899532999843359, 'forest': 0.951154887676239, 'glacier': 0.002136416733264923, 'mountain': 0.0009306151187047362, 'sea': 0.0029270262457430363, 'street': 0.03795143961906433}


array([[-1.1145415,  4.1539955, -1.9445516, -2.775591 , -1.6296945,
         0.932626 ]], dtype=float32)

The model has two flavors. `pytorch` gives the PyTorch network (`mlflow.pytorch.load_model`), which takes and returns tensors. `python_function` is the interface common to every MLflow model (`mlflow.pyfunc.load_model`), whose `predict` takes and returns NumPy arrays: it is what `mlflow models serve` serves, without the caller needing PyTorch. Neither transforms the image: `get_transform`'s preprocessing is still to do, exactly as in training. On a GPU, `pyfunc` would load the model there, which a model in MLflow 3's pt2 format (`torch.export`) refuses: it keeps the device it was saved on, here the CPU, hence `model_config={"device": "cpu"}`.

### Adding aliases and tags

Aliases and tags organize the registry's models. An alias names one precise version, and moves from one version to another; a tag goes on a model or on one of its versions. We give the `champion` alias to the latest version, and put two tags:

In [10]:
client.set_registered_model_alias(MODEL_NAME, "champion", version)
client.set_registered_model_tag(MODEL_NAME, "domain", "cv")
client.set_model_version_tag(MODEL_NAME, version, "size", "small")
client.get_model_version_by_alias(MODEL_NAME, "champion")

<ModelVersion: aliases=['champion'], creation_timestamp=1794002542081, current_stage='None', deployment_job_state=None, description='', last_updated_timestamp=1794002542081, metrics=None, model_id=None, name='dev.ml.landscape-classifier', params=None, run_id='d635b6d9ab2f4218ae17087531e884d6', run_link='', source='models:/m-4ce5c0b8364144f4aff80089d4e5ee9c', status='READY', status_message=None, tags={'size': 'small'}, user_id='', version='1', workspace='default'>

We can then load the model by its alias rather than by its version number:

In [11]:
model = mlflow.pytorch.load_model(f"models:/{MODEL_NAME}@champion")
with torch.inference_mode():
  print(model(batch).softmax(dim=1)[0])

tensor([4.8995e-03, 9.5115e-01, 2.1364e-03, 9.3062e-04, 2.9270e-03, 3.7951e-02])


## Creating an API with FastAPI

The API loads the model once, at startup (`lifespan`), from the registry and by its alias. `POST /predict` receives an image, transforms it with `get_transform`, the training's own function, and returns the most likely class and each class's probability; `GET /health` answers Kubernetes' probes:

In [12]:
%%writefile src/landscape_classifier/api.py
"""The classifier's HTTP API: `fastapi run src/landscape_classifier/api.py`."""

import os
from collections.abc import AsyncGenerator
from contextlib import asynccontextmanager
from typing import Any

import mlflow
import torch
from fastapi import FastAPI, UploadFile
from PIL import Image
from pydantic import BaseModel

from landscape_classifier.data import IMAGE_SIZE, get_transform

MODEL_URI = os.environ.get("MODEL_URI", "models:/dev.ml.landscape-classifier@champion")
models: dict[str, Any] = {}


@asynccontextmanager
async def lifespan(app: FastAPI) -> AsyncGenerator[None]:
  models["classifier"] = mlflow.pytorch.load_model(MODEL_URI)
  models["classes"] = mlflow.models.get_model_info(MODEL_URI).metadata["classes"]
  yield


app = FastAPI(lifespan=lifespan)


# Kubernetes' probes: the API only answers once `lifespan` has loaded the model.
@app.get("/health")
def health() -> dict[str, str]:
  return {"status": "ok"}


class Prediction(BaseModel):
  label: str
  probabilities: dict[str, float]


@app.post("/predict")
def predict(file: UploadFile) -> Prediction:
  image = Image.open(file.file).convert("RGB")
  batch = get_transform(IMAGE_SIZE)(image).unsqueeze(0)
  with torch.inference_mode():
    scores = models["classifier"](batch).softmax(dim=1)[0]
  probabilities = dict(zip(models["classes"], scores.tolist(), strict=True))
  label = max(probabilities, key=probabilities.__getitem__)
  return Prediction(label=label, probabilities=probabilities)

Writing src/landscape_classifier/api.py


The preprocessing must be exactly the training's: the API imports `get_transform` and `IMAGE_SIZE` from `landscape_classifier.data` instead of rewriting them. A difference (size, pixel scaling, channel order) raises no error: it silently degrades the predictions.

### Testing the API

`TestClient` calls the application without a server, and the `with` block runs its `lifespan`, which loads the model:

In [13]:
import importlib

from fastapi.testclient import TestClient

import landscape_classifier.api

api = importlib.reload(landscape_classifier.api)  # rereads api.py if it changed
with TestClient(api.app) as test_client, open("forest1.jpg", "rb") as image:
  print(test_client.get("/health").json())
  response = test_client.post("/predict", files={"file": image})
print(response.status_code, response.json())

{'status': 'ok'}
200 {'label': 'forest', 'probabilities': {'buildings': 0.004899532999843359, 'forest': 0.951154887676239, 'glacier': 0.002136416733264923, 'mountain': 0.0009306151187047362, 'sea': 0.0029270262457430363, 'street': 0.03795143961906433}}


## Promoting a model

To promote a model from one environment to the next, we copy the wanted version to the next environment's model, here the staging one (`staging.`). The copy doesn't keep the aliases: we give it `champion`, then the API, restarted with `MODEL_URI=models:/staging.ml.landscape-classifier@champion`, serves that model without changing its code:

In [14]:
import os

STAGING_NAME = "staging.ml.landscape-classifier"
staging = client.copy_model_version(
  src_model_uri=f"models:/{MODEL_NAME}@champion",
  dst_name=STAGING_NAME,
)
print(staging.aliases)
client.set_registered_model_alias(STAGING_NAME, "champion", staging.version)

os.environ["MODEL_URI"] = f"models:/{STAGING_NAME}@champion"
api = importlib.reload(landscape_classifier.api)
with TestClient(api.app) as test_client, open("forest1.jpg", "rb") as image:
  response = test_client.post("/predict", files={"file": image})
print(response.status_code, response.json())
del os.environ["MODEL_URI"]

Successfully registered model 'staging.ml.landscape-classifier'.
Copied version '1' of model 'dev.ml.landscape-classifier' to version '1' of model 'staging.ml.landscape-classifier'.


[]


200 {'label': 'forest', 'probabilities': {'buildings': 0.004899532999843359, 'forest': 0.951154887676239, 'glacier': 0.002136416733264923, 'mountain': 0.0009306151187047362, 'sea': 0.0029270262457430363, 'street': 0.03795143961906433}}


## Protecting the API with a key

An API key protects `POST /predict`: the API reads it from the `API_KEY` environment variable and refuses to start without it (in `lifespan`), rather than staying open to everyone; a request without an `X-API-Key` header equal to its value gets a 401 error.

In [15]:
%%writefile src/landscape_classifier/api.py
"""The classifier's HTTP API: `fastapi run src/landscape_classifier/api.py`."""

import os
import secrets
from collections.abc import AsyncGenerator
from contextlib import asynccontextmanager
from typing import Annotated, Any

import mlflow
import torch
from fastapi import Depends, FastAPI, HTTPException, Security, UploadFile
from fastapi.security import APIKeyHeader
from PIL import Image
from pydantic import BaseModel

from landscape_classifier.data import IMAGE_SIZE, get_transform

MODEL_URI = os.environ.get("MODEL_URI", "models:/dev.ml.landscape-classifier@champion")
models: dict[str, Any] = {}


@asynccontextmanager
async def lifespan(app: FastAPI) -> AsyncGenerator[None]:
  if not os.environ.get("API_KEY"):
    msg = "API_KEY is not set: the API refuses to start without it"
    raise RuntimeError(msg)
  models["classifier"] = mlflow.pytorch.load_model(MODEL_URI)
  models["classes"] = mlflow.models.get_model_info(MODEL_URI).metadata["classes"]
  yield


app = FastAPI(lifespan=lifespan)
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)


def check_api_key(key: Annotated[str | None, Security(api_key_header)]) -> None:
  if not secrets.compare_digest(key or "", os.environ["API_KEY"]):
    raise HTTPException(status_code=401, detail="Invalid API key")


# Kubernetes' probes: the API only answers once `lifespan` has loaded the model.
@app.get("/health")
def health() -> dict[str, str]:
  return {"status": "ok"}


class Prediction(BaseModel):
  label: str
  probabilities: dict[str, float]


@app.post("/predict", dependencies=[Depends(check_api_key)])
def predict(file: UploadFile) -> Prediction:
  image = Image.open(file.file).convert("RGB")
  batch = get_transform(IMAGE_SIZE)(image).unsqueeze(0)
  with torch.inference_mode():
    scores = models["classifier"](batch).softmax(dim=1)[0]
  probabilities = dict(zip(models["classes"], scores.tolist(), strict=True))
  label = max(probabilities, key=probabilities.__getitem__)
  return Prediction(label=label, probabilities=probabilities)

Overwriting src/landscape_classifier/api.py


### Testing the key

Without `API_KEY`, the API refuses to start; with a key set, a request without the key is refused, and a request with the key accepted:

In [16]:
import os

api = importlib.reload(landscape_classifier.api)  # rereads api.py
try:
  with TestClient(api.app):
    pass
except RuntimeError as error:
  print("Without API_KEY:", error)
else:
  print("Without API_KEY, the API started")

os.environ["API_KEY"] = "a-test-key"
with TestClient(api.app) as test_client:
  for headers in [{}, {"X-API-Key": "a-test-key"}]:
    with open("forest1.jpg", "rb") as image:
      response = test_client.post("/predict", files={"file": image}, headers=headers)
    print(headers, response.status_code, response.json())
del os.environ["API_KEY"]

Without API_KEY: API_KEY is not set: the API refuses to start without it


{} 401 {'detail': 'Invalid API key'}
{'X-API-Key': 'a-test-key'} 200 {'label': 'forest', 'probabilities': {'buildings': 0.004899532999843359, 'forest': 0.951154887676239, 'glacier': 0.002136416733264923, 'mountain': 0.0009306151187047362, 'sea': 0.0029270262457430363, 'street': 0.03795143961906433}}


## Creating a Docker image

The `Dockerfile` installs the dependencies before copying the code, so that Docker reuses their layer when only the code changes, and runs the server as a user who isn't `root`; the `.dockerignore` keeps the data and the credentials out of the build context:

In [17]:
%%writefile Dockerfile
FROM python:3.13-slim
COPY --from=ghcr.io/astral-sh/uv:0.13.1 /uv /uvx /bin/
WORKDIR /app
COPY pyproject.toml uv.lock README.md LICENSE ./
RUN uv sync --locked --no-install-project --no-dev --extra fastapi
COPY src ./src
RUN uv sync --locked --no-dev --extra fastapi
RUN useradd --create-home app
USER app
CMD ["uv", "run", "--no-sync", "fastapi", "run", "src/landscape_classifier/api.py"]

Writing Dockerfile


In [18]:
%%writefile .dockerignore
.env
*.env
.git
.venv
mlflow.db
mlruns
test-data
train-data

Writing .dockerignore


Docker isn't available in Colab. On a computer with Docker, with the MLflow credentials in a `creds.env` file (a `NAME=value` line for each of the `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME` and `MLFLOW_TRACKING_PASSWORD` variables):

    docker build -t landscape-classifier .
    docker run -p 8000:8000 --env-file creds.env -e API_KEY=a-test-key landscape-classifier
    curl -H "X-API-Key: a-test-key" -F file=@forest1.jpg http://localhost:8000/predict

`-p 8000:8000` forwards the host's port 8000 to the container's, `--env-file` passes the credentials at startup without putting them in the image, and `-e API_KEY=...` gives the API key, without which the API refuses to start.